# Segmentación de terreno con DeepLabV3 (local, MPS) — línea futura (E5)

Entrena un **DeepLabV3-ResNet50** (transfer learning) para predecir la máscara de terreno
a partir de la imagen NavCam, usando las máscaras de AI4Mars como etiquetas. Empezamos en
modo **binario** (roca = bedrock+big rock vs. resto), que está balanceado y alimenta la
cobertura (E1). Luego se puede extender a 4 clases (`BINARY = False`).

Cierra el círculo: **máscara predicha → nuestro pipeline → comparación con la anotación humana.**

**Requisitos:** `pip install torch torchvision` · Apple Silicon (`mps`). El entrenamiento
descarga los pesos pre-entrenados (~160 MB) la primera vez.


## 1. Configuración


In [ ]:
import sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "src").is_dir()), here)
sys.path.insert(0, str(root))
from src import config, mask_utils as mu, segmentation as seg

DEVICE   = "mps" if torch.backends.mps.is_available() else "cpu"
BINARY   = True     # True: roca/no-roca (2 clases) · False: 4 clases NAV
IMG_SIZE = 512
N_TRAIN, N_VAL, N_TEST = 2000, 400, 400
EPOCHS   = 6
BATCH    = 4
NUM_CLASSES = 2 if BINARY else 4
print("device:", DEVICE, "| binario:", BINARY, "| clases:", NUM_CLASSES)

## 2. Construir splits train/val/test
Se prioriza incluir imágenes con roca para que el modelo la vea.


In [ ]:
RESULTS = pd.read_csv(root / "outputs" / "results.csv")
pool = RESULTS[(RESULTS.frac_valid >= 0.2) &
               (RESULTS.quality_flag.isin(["ok", "no_bigrock", "no_rock"]))]
rng = np.random.default_rng(0)

rock_ids  = pool[pool.rock_coverage_pct.fillna(0) > 1].image_id.tolist()
other_ids = pool[pool.rock_coverage_pct.fillna(0) <= 1].image_id.tolist()
rng.shuffle(rock_ids); rng.shuffle(other_ids)

n_total = N_TRAIN + N_VAL + N_TEST
n_rock = min(len(rock_ids), n_total // 2)
ids = rock_ids[:n_rock] + other_ids[:n_total - n_rock]
rng.shuffle(ids); ids = ids[:n_total]

train_ids = ids[:N_TRAIN]; val_ids = ids[N_TRAIN:N_TRAIN+N_VAL]; test_ids = ids[N_TRAIN+N_VAL:]
print(f"train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")

## 3. Datasets y DataLoaders


In [ ]:
train_ds = seg.RockSegDataset(train_ids, IMG_SIZE, binary=BINARY)
val_ds   = seg.RockSegDataset(val_ids,   IMG_SIZE, binary=BINARY)
test_ds  = seg.RockSegDataset(test_ids,  IMG_SIZE, binary=BINARY)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
print("batches:", len(train_dl), "train /", len(val_dl), "val")

## 4. Modelo y entrenamiento
DeepLabV3-ResNet50 pre-entrenado, con la cabeza adaptada. Pérdida CrossEntropy
(ignora el valor 255). Se sigue el IoU de validación por época.


In [ ]:
model = seg.build_model(NUM_CLASSES, pretrained=True).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
crit = torch.nn.CrossEntropyLoss(ignore_index=255)

@torch.no_grad()
def eval_iou(dl):
    model.eval(); inter = np.zeros(NUM_CLASSES); union = np.zeros(NUM_CLASSES)
    for xb, yb in dl:
        xb = xb.to(DEVICE); pred = model(xb)["out"].argmax(1).cpu()
        v = yb != 255
        for c in range(NUM_CLASSES):
            p = (pred == c) & v; t = (yb == c) & v
            inter[c] += (p & t).sum().item(); union[c] += (p | t).sum().item()
    return [inter[c]/union[c] if union[c] else float("nan") for c in range(NUM_CLASSES)]

for ep in range(EPOCHS):
    model.train(); t0 = time.time(); running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        out = model(xb)
        loss = crit(out["out"], yb) + 0.4 * crit(out["aux"], yb)
        loss.backward(); opt.step(); running += loss.item()
    ious = eval_iou(val_dl)
    miou = np.nanmean(ious)
    print(f"época {ep+1}/{EPOCHS}  loss={running/len(train_dl):.3f}  "
          f"IoU_val={[f'{v:.2f}' for v in ious]}  mIoU={miou:.2f}  ({time.time()-t0:.0f}s)")

## 5. Evaluación en test (IoU por clase)


In [ ]:
test_dl = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2)
ious = eval_iou(test_dl)
names = ["no-roca", "roca"] if BINARY else ["soil", "bedrock", "sand", "big_rock"]
for n, v in zip(names, ious):
    print(f"  IoU {n:<9}: {v:.3f}")
print(f"  mIoU: {np.nanmean(ious):.3f}")

## 6. Cerrar el círculo: cobertura predicha vs. humana
Comparamos la cobertura de roca calculada sobre la máscara **predicha** vs. la **humana**,
sobre los píxeles que el humano etiquetó (para que sea comparable).


In [ ]:
import matplotlib.pyplot as plt
rows = []
for iid in test_ids[:120]:
    mp = config.MSL_NCAM_LABELS_TRAIN / f"{iid}.png"
    ip = mu.mask_to_image_path(mp)
    if ip is None: continue
    human = mu.read_mask(mp); valid = human != 255
    nv = valid.sum()
    if nv == 0: continue
    human_cov = 100 * np.isin(human, config.COVERAGE_CLASSES)[valid].mean()
    pred = seg.predict_mask(model, ip, IMG_SIZE, device=DEVICE)
    pred_rock = (pred == 1) if BINARY else np.isin(pred, config.COVERAGE_CLASSES)
    pred_cov = 100 * pred_rock[valid].mean()
    rows.append({"image_id": iid, "human_cov": human_cov, "pred_cov": pred_cov})

cov = pd.DataFrame(rows)
r = cov.human_cov.corr(cov.pred_cov)
mae = (cov.pred_cov - cov.human_cov).abs().mean()
print(f"n={len(cov)}  correlación cobertura r={r:.2f}  error abs medio={mae:.1f} pp")
fig, ax = plt.subplots(figsize=(6,6))
ax.plot([0,100],[0,100],"--",color="gray"); ax.scatter(cov.human_cov, cov.pred_cov, c="#2c6fbb", s=25, alpha=0.6)
ax.set(xlabel="cobertura humana (%)", ylabel="cobertura predicha (%)",
       title=f"Cobertura: DeepLabV3 vs humano (r={r:.2f})", xlim=(0,100), ylim=(0,100))
ax.grid(alpha=0.3); plt.show()

## 7. Guardar el modelo


In [ ]:
out = root / "outputs" / ("modelo_deeplab_binario.pt" if BINARY else "modelo_deeplab_4clases.pt")
torch.save(model.state_dict(), out)
print("modelo guardado en", out)

## 8. Interpretación y extensión

- **IoU de la clase roca** cuantifica qué tan bien un modelo aprendido reproduce la
  anotación humana; la **correlación de cobertura** muestra si sirve para el indicador E1.
- Para **4 clases**, pon `BINARY = False` y re-ejecuta; `big rock` tendrá IoU bajo por ser
  clase rara (documentar como limitación; considerar pesos de clase o *focal loss*).
- Extensión: alimentar las máscaras predichas al conteo (watershed) y comparar con las
  humanas, cerrando también E2.
